# pg_xarray Demo Book

A five-chapter tour: from "I have some scientific files" to "my Postgres is an ML feature store."

**Pre-requisites:**
1. Postgres + PostGIS + pg_xarray installed (`cargo pgrx install --features "reader-netcdf reader-grib"`).
2. The demo fixture files are committed under `demo/fixtures/` — no fixture-build step. The chapters hardcode the absolute path `/home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures`. If your checkout lives elsewhere, search-and-replace that path across all chapter cells before running them. The path must be absolute because `fs://` URIs are resolved by the Postgres backend, not the client.

**Source of truth:** the `.sql` files in this directory. This book is generated by `build_notebook.py` — re-run it after editing any chapter.

**Running cells:** install [jupysql] (`pip install jupysql sqlalchemy psycopg2-binary`) and run the setup cell below.

[jupysql]: https://jupysql.ploomber.io/

In [ ]:
# Optional: load jupysql to execute cells against Postgres.
# Skip this cell if you'd rather copy SQL into your own client.
%load_ext sql
%config SqlMagic.autopandas = True
%sql postgresql+psycopg2://postgres@localhost:5432/mydb


pg_xarray Demo · Chapter 0 — Setup

Loads the two extensions you need (PostGIS + pg_xarray). The demo fixture
files are committed alongside this notebook under demo/fixtures/, so there
is no fixture-build step. The chapters below hardcode the absolute path:
/home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures
If you cloned the repo somewhere else, search-and-replace that path
throughout the chapter cells before running them. The path has to be
absolute because `fs://` URIs are resolved by the Postgres backend, not
the client.

==> pg_xarray demo book — chapter 0: setup

In [ ]:
CREATE EXTENSION IF NOT EXISTS postgis;
CREATE EXTENSION IF NOT EXISTS pg_xarray;

== Catalog tables that pg_xarray manages ==

In [ ]:
SELECT table_name
FROM   information_schema.tables
WHERE  table_schema = 'pgx'
ORDER  BY table_name;

== Required SRF / FDW handlers ==

In [ ]:
SELECT routine_name
FROM   information_schema.routines
WHERE  routine_schema = 'pgx'
  AND  routine_name IN ('fetch', 'fetch_xyz', 'fetch_vec', 'fetch_xyz_vec',
                        'fetch_mesh', 'register_file', 'register_zarr_store',
                        'list_zarr_variables', 'fdw_handler')
ORDER  BY routine_name;

== Bundled fixtures (committed under demo/fixtures) ==
weather/          (Zarr v3 store, 3 data variables)
weather.nc        (NC3 — contiguous)
weather_chunked.nc (NC4 — HDF5 chunks; the 100 GB ERA5 path)
weather.grib2     (2-message GRIB2)
flood.slf         (4-node, 2-triangle SELAFIN)

pg_xarray Demo · Chapter 1 — Catalog a local file

One `pgx.register_file` call per (dataset, variable, file). The walker
reads the file's header / chunk index once, populates `pgx.chunks` with
per-chunk bbox + time + byte offsets, and writes CF metadata
(units, standard_name, scale_factor, ...) onto `pgx.variables`.
This works the same for Zarr, NetCDF (NC3 + NC4), and GRIB2.

==> Chapter 1: catalog a local file

1.1 — Zarr v3 (one chunk file per slab, one catalog row per chunk)

-- 1.1 Zarr v3 — store-level discovery + bulk register

In [ ]:
SELECT pgx.register_dataset('demo-weather-zarr', 'zarr');

See what's in the store WITHOUT registering anything yet.

In [ ]:
SELECT name, shape, dtype, is_data_variable
FROM   pgx.list_zarr_variables('fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/weather')
ORDER  BY name;

One call registers every data variable (skips coord axes like
latitude / longitude / level / valid_time).

In [ ]:
SELECT n_variables, n_chunks
FROM   pgx.register_zarr_store('demo-weather-zarr',
                               'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/weather',
                               lat_axis  := 'latitude',
                               lon_axis  := 'longitude',
                               time_axis := 'valid_time');

-- Catalog now sees the variables + their CF metadata:

In [ ]:
SELECT v.name, v.dtype, v.units, v.standard_name, v.scale_factor, v.add_offset
FROM   pgx.variables v JOIN pgx.datasets d ON d.id = v.dataset_id
WHERE  d.name = 'demo-weather-zarr'
ORDER  BY v.name;

1.2 — NetCDF-4 with HDF5 chunking (the path that makes 100 GB ERA5 tractable)

-- 1.2 NetCDF-4 — one catalog row per HDF5 chunk, with per-chunk bbox

In [ ]:
SELECT pgx.register_file(
    'demo-weather-nc', 't2m',
    'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/weather_chunked.nc', 'netcdf',
    lat_axis := 'latitude', lon_axis := 'longitude', time_axis := 'time'
);

-- Per-HDF5-chunk catalog rows — bbox split on longitude:

In [ ]:
SELECT chunk_key, bbox_wkt, time_lo
FROM   pgx.list_chunks('demo-weather-nc', 't2m')
ORDER  BY chunk_key
LIMIT  4;

1.3 — GRIB2 (one catalog row per message slab)

-- 1.3 GRIB2 — one catalog row per message slab

In [ ]:
SELECT pgx.register_file(
    'demo-weather-grib', 'TMP',
    'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/weather.grib2', 'grib2'
);

-- Per-message catalog rows with byte_offset + byte_length:

In [ ]:
SELECT chunk_key, bbox_wkt, time_lo, byte_offset, byte_length
FROM   pgx.list_chunks('demo-weather-grib', 'TMP')
ORDER  BY byte_offset;

1.4 — Query everything the same way: pgx.fetch

-- 1.4 pgx.fetch — same SRF for every format

-- 4 cells around (lat=51, lon=1) at level=500 hPa, t=02:00 from the Zarr store:

In [ ]:
SELECT lat, lon, level, time, value::numeric(7,3) AS t2m_K
FROM   pgx.fetch('demo-weather-zarr', 't2m_packed',
                 bbox_wkt   := 'POLYGON((0.5 50.5, 1.5 50.5, 1.5 51.5, 0.5 51.5, 0.5 50.5))',
                 level_from := 500, level_to := 500,
                 at_time    := '2024-01-01 02:00:00+00'::timestamptz);

-- A single cell from the GRIB2 file:

In [ ]:
SELECT lat, lon, value::numeric(7,3) AS t_K, time
FROM   pgx.fetch('demo-weather-grib', 'TMP',
                 at_time := '2024-01-01 03:00:00+00'::timestamptz)
ORDER  BY lat, lon
LIMIT  4;

== Try this ==
-- dataset summary (variable / chunk counts + extents):
"  SELECT * FROM pgx.dataset_summary('demo-weather-zarr');"
-- inspect one chunks bbox + byte range:
"  SELECT * FROM pgx.list_chunks('demo-weather-grib', 'TMP');"

pg_xarray Demo · Chapter 2 — Cloud-native: point at a public bucket

Same workflow as Chapter 1, except the URI is `http://`, `https://`,
`s3://`, or `gs://`. The file never lands on local disk:
pgx.register_file → walker reads the header bytes via OpenDAL range
GETs and writes the catalog
pgx.fetch         → predicate-pruned chunks → range GETs for just
the bytes the query needs
This chapter has TWO runnable demos:
* 2.A — works offline: use the localhost HTTP server that test.sh
spins up (or run `python3 -m http.server` over the bundled
demo/fixtures directory).
* 2.B — live against NOAA GFS on AWS Open Data (public HTTPS, no
credentials). A ~42 MB initial download at register_file
time, then pgx.fetch range-GETs only the bytes the query
touches.

==> Chapter 2: cloud-native register_file

2.A — register a GRIB2 file via HTTP (run python3 -m http.server first!)

-- 2.A Register via HTTP — start a fixture server in another terminal:
--
--     cd /home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures && python3 -m http.server 29980
--
-- then this works — no local file step needed:

In [ ]:
SELECT pgx.register_file(
    'demo-grib-http', 'TMP',
    'http://127.0.0.1:29980/weather.grib2', 'grib2'
);

-- The URI in the catalog now points at the HTTP server:

In [ ]:
SELECT chunk_key, uri, byte_offset, byte_length
FROM   pgx.list_chunks('demo-grib-http', 'TMP')
ORDER  BY byte_offset;

-- pgx.fetch range-GETs only the matching message bytes:

In [ ]:
SELECT lat, lon, time, value::numeric(7,3) AS t_K
FROM   pgx.fetch('demo-grib-http', 'TMP',
                 at_time := '2024-01-01 00:00:00+00'::timestamptz)
LIMIT  4;

2.B — NOAA GFS on AWS Open Data: a real public forecast over HTTPS

-- 2.B NOAA GFS (1°) — public HTTPS to noaa-gfs-bdp-pds, no AWS creds.
--     ~42 MB/file. register_file does one full GET to walk the message
--     index; pgx.fetch then range-GETs only the matching bytes.

In [ ]:
SELECT pgx.register_dataset('noaa-gfs-tmp', 'grib2');

Yesterday's 00z 1-degree GFS forecast at hour 0 (initial conditions).
NOAA keeps ~10 days of runs online. `current_date - 1` is safe — today's
run may not be uploaded yet depending on the hour you run this. Variable
'TMP' matches every Temperature message in the file (every level,
every timestep) — gribberish recognises this as NCEP's standard abbrev
for GRIB2 parameter (0,0,0).

In [ ]:
SELECT pgx.register_file(
    'noaa-gfs-tmp', 'TMP',
    'https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.'
        || to_char(current_date - 1, 'YYYYMMDD')
        || '/00/atmos/gfs.t00z.pgrb2.1p00.f000',
    'grib2'
);

-- Catalog rows are tiny: one per matched message, with byte_offset/length:

In [ ]:
SELECT chunk_key, time_lo, byte_offset, byte_length
FROM   pgx.list_chunks('noaa-gfs-tmp', 'TMP')
ORDER  BY byte_offset
LIMIT  5;

-- A bbox around London at 1000 hPa — pgx.fetch range-GETs just one message:
Three predicates push into the catalog: bbox (lat/lon), at_time, and the
level filter. Without `level_from`/`level_to`, fetch would return TMP at
every level the file carries (surface, 2 m, every pressure surface),
and the value column can spill past numeric(6,2) on missing-value
sentinels — so we use plain `numeric` and round for display.

In [ ]:
SELECT lat, lon, level, time, round(value::numeric, 2) AS t_K
FROM   pgx.fetch('noaa-gfs-tmp', 'TMP',
                 bbox_wkt   := 'POLYGON((-0.5 51.0, 0.5 51.0, 0.5 52.0, -0.5 52.0, -0.5 51.0))',
                 level_from := 1000, level_to := 1000,
                 at_time    := ((current_date - 1) || ' 00:00:00+00')::timestamptz)
ORDER  BY lat, lon
LIMIT  4;

More public datasets (commented — uncomment after wiring credentials)

== Try this ==
-- (A) ECMWF Open Data — public HTTPS, no credentials. ~140 MB/file at 0.25°.
"--     URL pattern; ECMWF retains the last ~5 days. gribberish maps every"
"--     ECMWF parameter via its discipline/category/number, so the variable"
"--     name follows NCEP convention ('PRES' for pressure, 'TMP' for temp)."
"--     SELECT pgx.register_file("
"--       'ecmwf-open-pres', 'PRES',"
"--       'https://data.ecmwf.int/forecasts/' || to_char(current_date, 'YYYYMMDD')"
"--         || '/00z/ifs/0p25/oper/' || to_char(current_date, 'YYYYMMDD')"
"--         || '000000-0h-oper-fc.grib2',"
"--       'grib2'"
"--     );"
--
-- (B) NOAA GFS at 0.25° — same workflow as 2.B but ~500 MB/file (higher res).
"--     SELECT pgx.register_file("
"--       'noaa-gfs-hires', 'TMP',"
"--       'https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs.' || to_char(current_date - 1, 'YYYYMMDD')"
"--         || '/00/atmos/gfs.t00z.pgrb2.0p25.f000',"
"--       'grib2'"
"--     );"
--
-- (C) Pangeo ARCO-ERA5 — a public Zarr v3 store on Google Cloud.
"--     gs://gcp-public-data-arco-era5/... — register_zarr_store enumerates"
--     hundreds of variables in one call. Use generously — small files,
--     large catalog, near-zero ingress cost.
"--     SELECT n_variables, n_chunks"
"--     FROM   pgx.register_zarr_store("
"--       'arco-era5',"
"--       'gs://gcp-public-data-arco-era5/raw/.../single-level.zarr');"
-- Once registered, pgx.fetch / pgx.fetch_xyz / pgx.fetch_vec /
-- pgx.fetch_mesh and the FDW all work the same as for local files.

pg_xarray Demo · Chapter 3 — FDW and JOIN pushdown

`CREATE FOREIGN TABLE … SERVER pgx_fdw OPTIONS (dataset 'x', variable 'y')`
gives you a regular-table façade over a (dataset, variable). WHERE
clauses on `lat` / `lon` / `level` / `time` push into the catalog as
bbox + range predicates. JOIN clauses against a regular PG table
push too — the planner picks a Nested Loop with the FDW as the
parameterized inner side, and runtime values from the outer row are
bound into the per-loop fetch.
Pre-requisite: Chapter 1 was run (catalog has demo-weather-zarr).

==> Chapter 3: FDW + JOIN pushdown

One server per (FDW, schema). Idempotent.

In [ ]:
DROP SERVER IF EXISTS demo_pgx CASCADE;
CREATE SERVER demo_pgx FOREIGN DATA WRAPPER pgx_fdw;

In [ ]:
DROP FOREIGN TABLE IF EXISTS demo_wx_t2m;
CREATE FOREIGN TABLE demo_wx_t2m (
    lat   DOUBLE PRECISION,
    lon   DOUBLE PRECISION,
    level DOUBLE PRECISION,
    "time" TIMESTAMPTZ,
    value DOUBLE PRECISION
) SERVER demo_pgx
  OPTIONS (dataset 'demo-weather-zarr', variable 't2m_packed');

3.1 — Basic WHERE pushdown

-- 3.1 WHERE pushdown — bbox + time + level are folded into the catalog query

In [ ]:
EXPLAIN (COSTS OFF, VERBOSE)
SELECT lat, lon, value
FROM   demo_wx_t2m
WHERE  lat BETWEEN 50 AND 51
  AND  lon BETWEEN 0 AND 2
  AND  level = 500
  AND  "time" = '2024-01-01 02:00:00+00'::timestamptz;

In [ ]:
SELECT lat, lon, value::numeric(7,3) AS t_K
FROM   demo_wx_t2m
WHERE  lat BETWEEN 50 AND 51
  AND  lon BETWEEN 0 AND 2
  AND  level = 500
  AND  "time" = '2024-01-01 02:00:00+00'::timestamptz
ORDER  BY lat, lon;

3.2 — JOIN pushdown with a stations table (the typical real workload)

-- 3.2 Per-station fetch via JOIN — runtime parameters bound from outer row

Set up a tiny weather-station inventory. In real life this is your
live sensor catalog.

In [ ]:
DROP TABLE IF EXISTS demo_stations;
CREATE TABLE demo_stations (
    station_id INT PRIMARY KEY,
    name       TEXT,
    lat        DOUBLE PRECISION,
    lon        DOUBLE PRECISION
);
INSERT INTO demo_stations VALUES
    (1, 'station-50-0', 50, 0),
    (2, 'station-51-1', 51, 1),
    (3, 'station-52-3', 52, 3);

In [ ]:
BEGIN;

Force the planner to use the parameterized Nested Loop so this
chapter actually demonstrates pushdown (with these costs PG will
almost always pick it anyway on a 3-row stations table).

In [ ]:
SET LOCAL enable_mergejoin = off;
SET LOCAL enable_hashjoin  = off;

-- EXPLAIN (the FDW becomes the parameterized inner of a Nested Loop):

In [ ]:
EXPLAIN (COSTS OFF)
SELECT s.name, f.value::numeric(7,3) AS t_K
FROM   demo_stations s
JOIN   demo_wx_t2m   f ON f.lat = s.lat AND f.lon = s.lon
WHERE  f.level = 500
  AND  f."time" = '2024-01-01 02:00:00+00'::timestamptz;

-- One row per station with the temperature at that station:

In [ ]:
SELECT s.name, f.value::numeric(7,3) AS t_K
FROM   demo_stations s
JOIN   demo_wx_t2m   f ON f.lat = s.lat AND f.lon = s.lon
WHERE  f.level = 500
  AND  f."time" = '2024-01-01 02:00:00+00'::timestamptz
ORDER  BY s.station_id;
COMMIT;

== Try this ==
-- "Local mean around each station" — JOIN with a bbox per station:
SELECT s.name, avg(f.value)::numeric(7,3) AS t_avg_K, count(*) AS n
FROM   demo_stations s,
LATERAL pgx.fetch(
"           'demo-weather-zarr', 't2m_packed',"
bbox_wkt := ST_AsText(ST_Buffer(
ST_SetSRID(ST_MakePoint(s.lon, s.lat), 4326), 0.5)),
"           level_from := 500, level_to := 500,"
"           at_time    := '2024-01-01 02:00:00+00'::timestamptz) f"
GROUP  BY s.name
ORDER  BY s.name;

pg_xarray Demo · Chapter 4 — Materialized views as a feature store

This is where the architecture pays off. Once a forecast file is
cataloged, you express "per-station, per-time-step features for ML"
as a normal SQL view. `REFRESH MATERIALIZED VIEW CONCURRENTLY` after
the daily ingest, and your training corpus is always fresh.
Pre-requisite: Chapters 1 + 3 were run (catalog has demo-weather-zarr +
demo_stations).

==> Chapter 4: per-station feature views

4.1 — Local-mean / range / count around each station

-- 4.1 Local statistics — "what was the temperature pattern around each station?"

In [ ]:
DROP MATERIALIZED VIEW IF EXISTS demo_features_local;
CREATE MATERIALIZED VIEW demo_features_local AS
SELECT
    s.station_id,
    s.name,
    f.time                                                  AS forecast_ts,
    avg(f.value)                                            AS t2m_mean_K,
    (max(f.value) - min(f.value))                           AS t2m_range_K,
    stddev_samp(f.value)                                    AS t2m_stddev,
    count(*)                                                AS n_cells
FROM demo_stations s,
LATERAL pgx.fetch(
    'demo-weather-zarr', 't2m_packed',
    bbox_wkt   := ST_AsText(ST_Buffer(
                    ST_SetSRID(ST_MakePoint(s.lon, s.lat), 4326),
                    1.0)),  -- ~110 km buffer
    level_from := 500, level_to := 500
) f
GROUP BY s.station_id, s.name, f.time
ORDER BY s.station_id, f.time;

In [ ]:
SELECT * FROM demo_features_local;

4.2 — Vertical profile features (uses level pushdown)

-- 4.2 Vertical features — temperature at multiple pressure levels

In [ ]:
DROP MATERIALIZED VIEW IF EXISTS demo_features_vertical;
CREATE MATERIALIZED VIEW demo_features_vertical AS
SELECT
    s.station_id,
    f.time                                                       AS forecast_ts,
    avg(f.value) FILTER (WHERE f.level = 1000)                   AS t_1000hPa,
    avg(f.value) FILTER (WHERE f.level =  850)                   AS t_850hPa,
    avg(f.value) FILTER (WHERE f.level =  500)                   AS t_500hPa,
    avg(f.value) FILTER (WHERE f.level = 1000)
        - avg(f.value) FILTER (WHERE f.level = 500)              AS lapse_K
FROM demo_stations s,
LATERAL pgx.fetch(
    'demo-weather-zarr', 't2m_packed',
    bbox_wkt   := ST_AsText(ST_Buffer(
                    ST_SetSRID(ST_MakePoint(s.lon, s.lat), 4326),
                    1.0)),
    level_from := 500, level_to := 1000
) f
GROUP BY s.station_id, f.time
ORDER BY s.station_id, f.time;

In [ ]:
SELECT station_id, forecast_ts,
       t_1000hPa::numeric(7,2),
       t_850hPa::numeric(7,2),
       t_500hPa::numeric(7,2),
       lapse_K::numeric(7,2)
FROM   demo_features_vertical;

4.3 — A "training-set" view: would join your observations table here

-- 4.3 Training set shape — observations LEFT JOIN forecast features

Synthetic observations table — in real life this is your live sensor feed.

In [ ]:
DROP TABLE IF EXISTS demo_observations;
CREATE TABLE demo_observations (
    station_id INT,
    obs_ts     TIMESTAMPTZ,
    observed_K DOUBLE PRECISION
);
INSERT INTO demo_observations VALUES
    (1, '2024-01-01 01:00:00+00', 274.95),
    (1, '2024-01-01 02:00:00+00', 275.10),
    (2, '2024-01-01 02:00:00+00', 275.42),
    (3, '2024-01-01 02:00:00+00', 275.51);

-- The shape an ML model would consume:

In [ ]:
SELECT
    o.station_id,
    o.obs_ts,
    o.observed_K,
    l.t2m_mean_K::numeric(7,2)  AS forecast_local_mean,
    l.t2m_range_K::numeric(7,2) AS forecast_local_range,
    v.lapse_K::numeric(7,2)     AS forecast_lapse,
    (o.observed_K - l.t2m_mean_K)::numeric(7,2) AS forecast_error
FROM demo_observations o
LEFT JOIN demo_features_local    l ON (o.station_id, o.obs_ts) = (l.station_id, l.forecast_ts)
LEFT JOIN demo_features_vertical v ON (o.station_id, o.obs_ts) = (v.station_id, v.forecast_ts)
ORDER BY o.station_id, o.obs_ts;

== Try this ==
-- After your daily ingest:
REFRESH MATERIALIZED VIEW CONCURRENTLY demo_features_local;
REFRESH MATERIALIZED VIEW CONCURRENTLY demo_features_vertical;
-- Or hook pg_ml in for in-DB training:
"--     SELECT pg_ml.train('temperature_model',"
"--                        table_name := 'training_set',"
"--                        target     := 'observed_K');"

pg_xarray Demo · Chapter 5 — Unstructured meshes (SELAFIN / TELEMAC)

Hydraulic / hydrological / FEM data lives on unstructured meshes —
triangles or polygons, not a regular lat/lon grid. pg_xarray indexes
them in `pgx.meshes` / `pgx.mesh_versions` / `pgx.mesh_nodes` /
`pgx.mesh_cells`, and `pgx.fetch_mesh(dataset, variable, ...)` joins
chunk values to the mesh geometry.
This chapter uses the SELAFIN fixture from make_fixture.py — a tiny
4-node, 2-triangle mesh with WATER DEPTH + VELOCITY U at two
timesteps. The same `pgx.register_file` call enumerates messages /
chunks / messages, auto-creates the mesh + version, and populates
nodes + cells from the file's X/Y/IKLE arrays.

==> Chapter 5: unstructured mesh (SELAFIN)

5.1 — Register a SELAFIN file (auto-creates mesh + version + nodes + cells)

-- 5.1 Register the file — one call per variable; the mesh is created once

In [ ]:
SELECT pgx.register_dataset('demo-flood', 'selafin', default_srid := 0);
SELECT pgx.register_file('demo-flood', 'WATER DEPTH', 'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/flood.slf', 'selafin');
SELECT pgx.register_file('demo-flood', 'VELOCITY U',  'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/flood.slf', 'selafin');

5.2 — Inspect the mesh

-- 5.2 Mesh nodes (X/Y → PostGIS Point):

In [ ]:
SELECT mn.node_id, ST_AsText(mn.geom) AS geom
FROM   pgx.mesh_nodes mn
JOIN   pgx.mesh_versions mv ON mv.id = mn.mesh_version_id
JOIN   pgx.meshes        m  ON m.id  = mv.mesh_id
JOIN   pgx.datasets      d  ON d.id  = m.dataset_id
WHERE  d.name = 'demo-flood'
ORDER  BY mn.node_id;

-- Mesh cells (triangles with node_ids[] connectivity + centroid):

In [ ]:
SELECT mc.cell_id, mc.node_ids, ST_AsText(mc.centroid) AS centroid
FROM   pgx.mesh_cells mc
JOIN   pgx.mesh_versions mv ON mv.id = mc.mesh_version_id
JOIN   pgx.meshes        m  ON m.id  = mv.mesh_id
JOIN   pgx.datasets      d  ON d.id  = m.dataset_id
WHERE  d.name = 'demo-flood'
ORDER  BY mc.cell_id;

5.3 — Fetch per-node values at a specific timestep

-- 5.3 fetch_mesh — values joined to node geometry

-- WATER DEPTH at t=0:

In [ ]:
SELECT node_id, geom_wkt, value::numeric(6,2) AS depth_m
FROM   pgx.fetch_mesh('demo-flood', 'WATER DEPTH',
                      at_time := '2024-01-01 00:00:00+00'::timestamptz)
ORDER  BY node_id;

-- VELOCITY U at t=01:00 — uniform 1.0 m/s in this fixture:

In [ ]:
SELECT node_id, geom_wkt, value::numeric(6,2) AS vel_u_mps
FROM   pgx.fetch_mesh('demo-flood', 'VELOCITY U',
                      at_time := '2024-01-01 01:00:00+00'::timestamptz)
ORDER  BY node_id;

5.4 — Spatial query: WATER DEPTH inside a bbox

-- 5.4 Bbox prune — only nodes inside the requested envelope

In [ ]:
SELECT node_id, geom_wkt, value::numeric(6,2) AS depth_m
FROM   pgx.fetch_mesh('demo-flood', 'WATER DEPTH',
                      bbox_wkt := 'POLYGON((-0.1 -0.1, 1.1 -0.1, 1.1 0.5, -0.1 0.5, -0.1 -0.1))',
                      at_time  := '2024-01-01 00:00:00+00'::timestamptz)
ORDER  BY node_id;

5.5 — Export the animated water surface as glTF Binary (GLB)

-- 5.5 Visualisation — emit an animated GLB of WATER DEPTH + flow arrows.
--      Open the resulting file in https://gltf-viewer.donmccurdy.com/
--      or the Khronos glTF Sample Viewer to see the colour-graded
--      surface morph between timesteps with VELOCITY U as flow arrows.

Surface coloured by WATER DEPTH, flow arrows from VELOCITY U.
z_scale exaggerates the displacement so the morph is visible at this fixture's
(4-node) scale; in production tune to taste or pass 1.0.
-- GLB byte length:

In [ ]:
SELECT length(pgx.xarray_to_glb(
    'demo-flood',
    'WATER DEPTH',
    flow_uv  => ARRAY['VELOCITY U'],
    z_scale  => 10.0,
    colormap => 'viridis'
)) AS glb_bytes;

The asset.extras block carries dataset / colormap / vmin / vmax so an
external viewer can render a legend without round-tripping to the catalog.
Write the bytes to /tmp via \lo_export-style escape if your psql supports it;
otherwise pipe via COPY ... TO PROGRAM or a small client.
== Try this ==
-- Write the GLB to disk (psql >= 16):
"  \\copy (SELECT pgx.xarray_to_glb('demo-flood', 'WATER DEPTH',"
"                                   flow_uv => ARRAY['VELOCITY U'],"
"                                   z_scale => 10.0)) TO '/tmp/flood.glb' (FORMAT binary);"
-- Then open /tmp/flood.glb at https://gltf-viewer.donmccurdy.com/

== Try this ==
-- Cell-level features: average WATER DEPTH per triangle (manual join):
SELECT mc.cell_id,
ST_AsText(mc.centroid) AS centroid,
avg(v.value)::numeric(6,2) AS mean_depth_m
"  FROM   pgx.fetch_mesh('demo-flood', 'WATER DEPTH',"
"                        at_time := '2024-01-01 00:00:00+00'::timestamptz) v"
JOIN   pgx.mesh_cells mc ON v.node_id = ANY(mc.node_ids)
GROUP  BY mc.cell_id, mc.centroid
ORDER  BY mc.cell_id;
-- The catalog model is the same for FEM exodus / ParaView XDMF / MED —
-- each format is one walker function returning (nodes, cells, chunks).

5.6 — A meatier SELAFIN: register, query a subset via the index, then GLB.

`flood.slf` is a 4-node toy — useful for round-trip tests, less so for
visual demos. Here we generate a 231-node / 400-triangle / 30-timestep
synthetic SELAFIN that simulates a sine wave propagating through a
200 m × 100 m channel. Real TELEMAC datasets (Malpasset dam break etc.)
have the same shape; swap the `\!` line below for a `curl` against any
open-data SELAFIN you have access to.
The narrative of this step:
1. Generate (or download) the file
2. Register it as a dataset — pgx.register_file does the mesh discovery
3. Use the catalog index to filter the spatial subset we care about
4. Export that subset to GLB and open it in a viewer

==> Chapter 5.6: 231-node wave channel — generate, subset, export GLB

5.6.1 — Build the fixture (one-shot; idempotent, file is cached on disk).
-- 5.6.1 Generate wave_channel.slf (idempotent)

5.6.2 — Register each variable. Same pattern as 5.1 — the mesh is
created once on the first call, subsequent calls share it.
-- 5.6.2 Register the wave-channel SELAFIN

In [ ]:
SELECT pgx.register_dataset('wave-channel', 'selafin', default_srid := 0);
SELECT pgx.register_file(
    'wave-channel', 'WATER DEPTH',
    'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/wave_channel.slf',
    'selafin'
);
SELECT pgx.register_file(
    'wave-channel', 'VELOCITY U',
    'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/wave_channel.slf',
    'selafin'
);
SELECT pgx.register_file(
    'wave-channel', 'VELOCITY V',
    'fs:///home/ubuntu/dev/pg_extensions/extensions/pg_xarray/demo/fixtures/wave_channel.slf',
    'selafin'
);

5.6.3 — Demonstrate index pruning. The mesh covers x ∈ [0, 200], y ∈ [0, 100].
A bbox of x ∈ [50, 150], y ∈ [20, 80] should keep ~half the nodes.
-- 5.6.3 Index-pruned subset query — node counts inside vs outside bbox:

In [ ]:
SELECT
    count(*) FILTER (WHERE in_bbox) AS nodes_inside,
    count(*) FILTER (WHERE NOT in_bbox) AS nodes_outside
FROM (
    SELECT
        public.ST_X(geom) BETWEEN 50 AND 150
        AND public.ST_Y(geom) BETWEEN 20 AND 80 AS in_bbox
    FROM   pgx.mesh_nodes mn
    JOIN   pgx.mesh_versions mv ON mv.id = mn.mesh_version_id
    JOIN   pgx.meshes m  ON m.id = mv.mesh_id
    JOIN   pgx.datasets d ON d.id = m.dataset_id
    WHERE  d.name = 'wave-channel'
) t;

The same bbox applied to pgx.fetch_mesh — the index does the pruning
before the chunk-decode step (see [src/srf/fetch_mesh.rs] for the JOIN
to mesh_nodes that drops rows outside the bbox).
-- WATER DEPTH samples inside the bbox at t=600s (5 rows):

In [ ]:
SELECT node_id, geom_wkt, value::numeric(6,3) AS depth_m
FROM   pgx.fetch_mesh(
           'wave-channel', 'WATER DEPTH',
           at_time  := '2024-06-01 00:10:00+00'::timestamptz,
           bbox_wkt := 'POLYGON((50 20, 150 20, 150 80, 50 80, 50 20))'
       )
ORDER  BY node_id
LIMIT  5;

5.6.4 — Translate the bbox subset to an animated GLB.
-- 5.6.4 Export the bbox subset to an animated GLB
--   • surface_var = WATER DEPTH (drives Z displacement + colour)
--   • flow_uv     = (VELOCITY U, VELOCITY V) — animated LINES arrows
--   • bbox_wkt    = 100 m × 60 m window centred on the channel
--   • z_scale     = 5 — exaggerate so the wave is visible at this scale
--   • time_scale  = 60 — sim time / 60 → 29 minutes of physics plays in 29 s

In [ ]:
SELECT length(pgx.xarray_to_glb(
    'wave-channel',
    'WATER DEPTH',
    flow_uv  => ARRAY['VELOCITY U', 'VELOCITY V'],
    bbox_wkt => 'POLYGON((50 20, 150 20, 150 80, 50 80, 50 20))',
    z_scale  => 5.0,
    colormap => 'viridis',
    options  => '{"arrow_scale": 8.0, "time_scale": 60}'::jsonb
)) AS bbox_glb_bytes;

== Try this ==
-- Save the full-mesh + bbox-subset GLBs side-by-side and compare:
"  \\copy (SELECT pgx.xarray_to_glb('wave-channel', 'WATER DEPTH',"
"                                   flow_uv => ARRAY['VELOCITY U', 'VELOCITY V'],"
"                                   z_scale => 5.0,"
"                                   options => '{\"arrow_scale\": 8.0, \"time_scale\": 60}'::jsonb))"
"        TO '/tmp/wave_channel_full.glb' (FORMAT binary);"
"  \\copy (SELECT pgx.xarray_to_glb('wave-channel', 'WATER DEPTH',"
"                                   flow_uv  => ARRAY['VELOCITY U', 'VELOCITY V'],"
"                                   bbox_wkt => 'POLYGON((50 20, 150 20, 150 80, 50 80, 50 20))',"
"                                   z_scale  => 5.0,"
"                                   options  => '{\"arrow_scale\": 8.0, \"time_scale\": 60}'::jsonb))"
"        TO '/tmp/wave_channel_bbox.glb' (FORMAT binary);"
-- Then open both at https://gltf-viewer.donmccurdy.com/ — the bbox
-- file should contain only the centre 100 m × 60 m strip of the channel,
-- proving the catalog index pruned the GLB output as well as the query.

5.7 — 2D raster + WMS endpoint (GIS-tooling compliant, no GeoServer)

Where the GLB pipeline above targets 3D viewers (three.js, glTF Sample
Viewer), this section covers the 2D side of the same data: PNG raster
output for dashboards, map portals, and GIS clients (QGIS, ArcGIS,
Leaflet, OpenLayers) that already speak OGC standards.
Two surfaces, same catalog:
1. `pgx.xarray_to_png(...)` — single-call raster export. Returns a
PNG bytea for the requested timestep / bbox. Composable in SQL.
2. WMS 1.3.0 HTTP endpoint — `pg_xarray.wms_enabled = on` in
postgresql.conf brings up a read-only HTTP server inside a
Postgres bgworker. QGIS adds it as a WMS layer directly; no
separate GeoServer / JVM in the loop.
The WMS server uses the same bbox + time + CRS plumbing the catalog
already exposes — `GetMap?BBOX=...&TIME=...` maps 1:1 onto the
`pgx.fetch_mesh` index. Pre-rendering / cache absorption is up to a
reverse proxy (every GetMap response carries `Cache-Control: max-age`).

==> Chapter 5.7: 2D raster + WMS endpoint

5.7.1 — Single PNG via SQL.
-- 5.7.1 Render a single timestep as PNG via SQL (no HTTP needed):

In [ ]:
SELECT length(pgx.xarray_to_png(
    'wave-channel',
    'WATER DEPTH',
    at_time  => '2024-06-01 00:10:00+00'::timestamptz,
    bbox_wkt => 'POLYGON((50 20, 150 20, 150 80, 50 80, 50 20))',
    width    => 800,
    height   => 400,
    colormap => 'viridis'
)) AS png_bytes;

== Try this ==
-- Save a PNG of the central strip at t=10min:
"  \\copy (SELECT pgx.xarray_to_png('wave-channel', 'WATER DEPTH',"
"             at_time  => '2024-06-01 00:10:00+00'::timestamptz,"
"             bbox_wkt => 'POLYGON((50 20, 150 20, 150 80, 50 80, 50 20))',"
"             width    => 800, height => 400)) TO '/tmp/wave_t10.png' (FORMAT binary);"

5.7.2 — WMS endpoint, no separate server.
-- 5.7.2 WMS 1.3.0 endpoint — pg_xarray IS the WMS server.
--
-- Required postgresql.conf:
--   shared_preload_libraries = ''pg_xarray''
--   pg_xarray.wms_enabled    = on
--   pg_xarray.wms_port       = 7800        # default
--   pg_xarray.wms_bind_host  = ''127.0.0.1''  # loopback only — put nginx in front
--   pg_xarray.wms_cache_seconds = 60       # Cache-Control max-age
--   pg_xarray.database       = ''pg_xarray_demo''  # whichever DB has the catalog
-- Then from the shell:
"--   curl 'http://localhost:7800/wms?SERVICE=WMS&REQUEST=GetCapabilities'"
"--   curl -o tile.png 'http://localhost:7800/wms?SERVICE=WMS&REQUEST=GetMap&LAYERS=wave-channel:WATER%20DEPTH&BBOX=0,0,200,100&WIDTH=800&HEIGHT=400&CRS=EPSG:0&FORMAT=image/png&TIME=2024-06-01T00:10:00Z'"
-- Or from QGIS: Layer → Add Layer → Add WMS Layer →
"--   New Connection → URL http://localhost:7800/wms → wave-channel:WATER DEPTH"
-- Performance note: every GetMap response carries Cache-Control:
-- max-age=N; put nginx/Varnish/CDN in front for steady-state read traffic.
-- Single bgworker today; for >1000 tiles/sec → PG read replicas.